In [ ]:
"""Graph Enrichment library for the Graph Augmented AI Workshop.

This module handles the enrichment pipeline:
- Resolving schema-level suggestions into instance-level proposals
- Filtering proposals by confidence level
- Writing approved proposals back to Neo4j with provenance
- Validating enrichment results

Called by the '8 - Graph Enrichment' notebook via %run.
"""

import json
import re
import time
from dataclasses import dataclass, field
from datetime import datetime, timezone
from enum import Enum
from typing import Any

In [ ]:
# ─── Schemas ─────────────────────────────────────────────────────────


class ConfidenceLevel(str, Enum):
    """Confidence level for an enrichment proposal."""
    HIGH = "high"
    MEDIUM = "medium"
    LOW = "low"


@dataclass
class NodeReference:
    """A reference to a specific node in the graph."""
    label: str
    key_property: str
    key_value: str


@dataclass
class EnrichmentProposal:
    """A concrete proposal to add a relationship between two specific nodes."""
    source_node: NodeReference
    relationship_type: str
    target_node: NodeReference
    confidence: ConfidenceLevel
    source_document: str
    extracted_phrase: str


@dataclass
class EnrichmentResult:
    """Result of filtering proposals by confidence."""
    proposals: list
    approved: list = field(default_factory=list)
    flagged: list = field(default_factory=list)
    rejected: list = field(default_factory=list)


@dataclass
class WriteBackReport:
    """Summary of what was written to Neo4j."""
    created: int = 0
    updated: int = 0
    skipped: list = field(default_factory=list)

In [ ]:
# ─── Resolver ────────────────────────────────────────────────────────

_RESOLUTION_PROMPT = """\
You are resolving schema-level graph enrichment suggestions into specific, \
instance-level proposals that can be written to Neo4j.

The analysis phase identified these suggested relationship types:

{suggestions}

For each suggested relationship type above, identify EVERY specific pair of \
nodes where evidence in the documents supports creating this relationship. \
For each pair, provide:
- The exact source node label, key property name, and key value
- The relationship type
- The exact target node label, key property name, and key value
- The confidence level: "high" if the document explicitly states it, \
"medium" if it is strongly implied, "low" if it is only loosely suggested
- The source document filename
- The exact quoted phrase from the document that supports this proposal

Return your answer as a JSON array of objects with this exact structure:
[
  {{
    "source_node": {{"label": "Customer", "key_property": "customerId", "key_value": "C0001"}},
    "relationship_type": "INTERESTED_IN",
    "target_node": {{"label": "Sector", "key_property": "name", "key_value": "Renewable Energy"}},
    "confidence": "high",
    "source_document": "customer_profile_001.html",
    "extracted_phrase": "expressed interest in expanding his portfolio to include renewable energy stocks"
  }}
]

Return ONLY the JSON array. No commentary, no markdown fencing.
"""


def _format_suggestions(relationships: list[dict]) -> str:
    """Format schema-level relationship suggestions into text."""
    lines = []
    for i, rel in enumerate(relationships, 1):
        src = rel.get("source_label", "?")
        rtype = rel.get("relationship_type", "?")
        tgt = rel.get("target_label", "?")
        lines.append(f"{i}. ({src})-[{rtype}]->({tgt})")
        if rel.get("description"):
            lines.append(f"   Description: {rel['description']}")
        if rel.get("source_evidence"):
            lines.append(f"   Evidence: {rel['source_evidence']}")
        if rel.get("example_instances"):
            lines.append(f"   Examples: {rel['example_instances']}")
        lines.append("")
    return "\n".join(lines)


def _strip_markdown_fence(text: str) -> str:
    """Remove markdown code fences from a response."""
    cleaned = text.strip()
    if "```" not in cleaned:
        return cleaned
    first = cleaned.find("```")
    last = cleaned.rfind("```")
    if first == last:
        return cleaned
    after_first = cleaned[first + 3:]
    content_start = after_first.find("\n")
    if content_start == -1:
        return cleaned
    inner = after_first[content_start + 1:]
    trailing_fence = inner.rfind("```")
    if trailing_fence == -1:
        return inner.strip()
    return inner[:trailing_fence].strip()


def _parse_proposals(text: str) -> list[EnrichmentProposal]:
    """Parse the supervisor's JSON response into EnrichmentProposal objects."""
    cleaned = _strip_markdown_fence(text)
    raw = json.loads(cleaned)
    if not isinstance(raw, list):
        raw = [raw]

    required_keys = {"label", "key_property", "key_value"}
    proposals = []

    for i, item in enumerate(raw):
        if not isinstance(item, dict):
            print(f"  [WARN] Item {i}: not a dict, skipping")
            continue

        source = item.get("source_node")
        target = item.get("target_node")
        rel_type = item.get("relationship_type")

        if not isinstance(source, dict) or not required_keys.issubset(source):
            print(f"  [WARN] Item {i}: invalid source_node, skipping")
            continue
        if not isinstance(target, dict) or not required_keys.issubset(target):
            print(f"  [WARN] Item {i}: invalid target_node, skipping")
            continue
        if not rel_type:
            print(f"  [WARN] Item {i}: missing relationship_type, skipping")
            continue

        conf_str = str(item.get("confidence", "medium")).lower()
        confidence = ConfidenceLevel(conf_str) if conf_str in ("high", "medium", "low") else ConfidenceLevel.MEDIUM

        proposals.append(EnrichmentProposal(
            source_node=NodeReference(
                label=source["label"],
                key_property=source["key_property"],
                key_value=str(source["key_value"]),
            ),
            relationship_type=rel_type,
            target_node=NodeReference(
                label=target["label"],
                key_property=target["key_property"],
                key_value=str(target["key_value"]),
            ),
            confidence=confidence,
            source_document=item.get("source_document", "unknown"),
            extracted_phrase=item.get("extracted_phrase", ""),
        ))

    return proposals


def _query_supervisor(endpoint_name: str, prompt: str) -> str:
    """Send a query to the Supervisor Agent endpoint."""
    from databricks_openai import DatabricksOpenAI

    client = DatabricksOpenAI()
    response = client.responses.create(
        model=endpoint_name,
        input=[{"role": "user", "content": prompt}],
    )
    return response.output[0].content[0].text


def resolve_proposals(
    suggested_relationships: list[dict],
    endpoint_name: str,
) -> list[EnrichmentProposal]:
    """Resolve schema-level suggestions into instance-level proposals.

    Calls the Supervisor Agent to cross-reference suggested relationship
    types against actual graph data and documents.

    Args:
        suggested_relationships: List of relationship dicts from Lab 7 JSON output.
        endpoint_name: Supervisor Agent serving endpoint name.

    Returns:
        List of concrete EnrichmentProposal instances.
    """
    if not suggested_relationships:
        print("  No suggested relationships to resolve.")
        return []

    print("\n" + "=" * 60)
    print("RESOLVING SCHEMA-LEVEL SUGGESTIONS INTO INSTANCE PROPOSALS")
    print("=" * 60)
    print(f"  Resolving {len(suggested_relationships)} suggested relationship types...")
    print("  Calling Supervisor Agent (this may take 1-3 minutes)...\n")

    suggestion_text = _format_suggestions(suggested_relationships)
    prompt = _RESOLUTION_PROMPT.format(suggestions=suggestion_text)

    t0 = time.time()
    raw_response = _query_supervisor(endpoint_name, prompt)
    elapsed = time.time() - t0

    try:
        proposals = _parse_proposals(raw_response)
        print(f"  [OK] {len(proposals)} instance-level proposals resolved in {elapsed:.1f}s")
    except (json.JSONDecodeError, TypeError) as e:
        print(f"  [WARN] Failed to parse resolver response: {e}")
        print(f"  Raw response preview: {raw_response[:500]}")
        proposals = []

    print("=" * 60)
    return proposals

In [ ]:
# ─── Confidence Filter ────────────────────────────────────────────────


def filter_proposals(proposals: list[EnrichmentProposal]) -> EnrichmentResult:
    """Sort proposals by confidence into decision buckets.

    - HIGH   -> auto-approve, proceed to write-back
    - MEDIUM -> approve with flag for human review, proceed to write-back
    - LOW    -> reject, report but do not write
    """
    approved, flagged, rejected = [], [], []

    for p in proposals:
        if p.confidence == ConfidenceLevel.HIGH:
            approved.append(p)
        elif p.confidence == ConfidenceLevel.MEDIUM:
            flagged.append(p)
        else:
            rejected.append(p)

    result = EnrichmentResult(
        proposals=proposals,
        approved=approved,
        flagged=flagged,
        rejected=rejected,
    )

    print("\n" + "=" * 60)
    print("CONFIDENCE FILTER RESULTS")
    print("=" * 60)
    print(f"  Total proposals:      {len(proposals)}")
    print(f"  AUTO-APPROVED (HIGH): {len(approved)}")
    print(f"  FLAGGED (MEDIUM):     {len(flagged)}")
    print(f"  REJECTED (LOW):       {len(rejected)}")

    def _fmt(p):
        return (f"    ({p.source_node.label}:{p.source_node.key_value})"
                f"-[{p.relationship_type}]->"
                f"({p.target_node.label}:{p.target_node.key_value})")

    if approved:
        print("\n  --- Auto-approved ---")
        for p in approved:
            print(_fmt(p))

    if flagged:
        print("\n  --- Flagged for review ---")
        for p in flagged:
            print(_fmt(p))

    if rejected:
        print("\n  --- Rejected ---")
        for p in rejected:
            print(_fmt(p))

    print("=" * 60)
    return result

In [ ]:
# ─── Write-back to Neo4j ──────────────────────────────────────────────

_CYPHER_ID_RE = re.compile(r"^[a-zA-Z_][a-zA-Z0-9_]*$")


def _validate_identifier(name: str, context: str) -> None:
    """Validate that a string is a safe Cypher identifier."""
    if not _CYPHER_ID_RE.match(name):
        raise ValueError(f"Invalid Cypher {context}: {name!r}")


_MERGE_QUERY = """\
MATCH (src:{source_label} {{{source_key}: $source_value}})
MATCH (tgt:{target_label} {{{target_key}: $target_value}})
MERGE (src)-[r:{relationship_type}]->(tgt)
ON CREATE SET
  r.confidence = $confidence,
  r.source_document = $source_document,
  r.extracted_phrase = $extracted_phrase,
  r.enriched_at = $enriched_at,
  r._created = true
ON MATCH SET
  r.confidence = $confidence,
  r.source_document = $source_document,
  r.extracted_phrase = $extracted_phrase,
  r.enriched_at = $enriched_at,
  r._created = false
RETURN r._created AS was_created
"""

_CHECK_NODE = """\
MATCH (n:{label} {{{key}: $value}})
RETURN count(n) > 0 AS exists
"""


def _build_merge_query(p: EnrichmentProposal) -> str:
    """Build a parameterized Cypher MERGE query for a proposal."""
    _validate_identifier(p.source_node.label, "source label")
    _validate_identifier(p.source_node.key_property, "source key")
    _validate_identifier(p.target_node.label, "target label")
    _validate_identifier(p.target_node.key_property, "target key")
    _validate_identifier(p.relationship_type, "relationship type")
    return _MERGE_QUERY.format(
        source_label=p.source_node.label,
        source_key=p.source_node.key_property,
        target_label=p.target_node.label,
        target_key=p.target_node.key_property,
        relationship_type=p.relationship_type,
    )


def _check_node_exists(tx, label, key, value) -> bool:
    _validate_identifier(label, "label")
    _validate_identifier(key, "key")
    query = _CHECK_NODE.format(label=label, key=key)
    result = tx.run(query, value=value)
    record = result.single()
    return record is not None and record["exists"]


def _execute_merge(tx, query, params):
    result = tx.run(query, **params)
    return result.single()


def write_proposals(
    proposals: list[EnrichmentProposal],
    neo4j_url: str,
    neo4j_user: str,
    neo4j_pass: str,
    dry_run: bool = False,
) -> WriteBackReport:
    """Write approved proposals to Neo4j.

    Uses MERGE for idempotency. Each relationship gets provenance properties:
    confidence, source_document, extracted_phrase, enriched_at.

    Args:
        proposals: Proposals that passed the confidence filter.
        neo4j_url: Neo4j connection URI.
        neo4j_user: Neo4j username.
        neo4j_pass: Neo4j password.
        dry_run: If True, report what would be written without executing.

    Returns:
        WriteBackReport summarizing the results.
    """
    if not proposals:
        print("  No proposals to write.")
        return WriteBackReport()

    print("\n" + "=" * 60)
    print(f"WRITE-BACK TO NEO4J{' (DRY RUN)' if dry_run else ''}")
    print("=" * 60)
    print(f"  Proposals to write: {len(proposals)}")

    if dry_run:
        print("\n  --- Dry run: no changes will be made ---")
        for p in proposals:
            print(
                f"  WOULD WRITE: ({p.source_node.label}:{p.source_node.key_value})"
                f"-[{p.relationship_type}]->"
                f"({p.target_node.label}:{p.target_node.key_value})"
                f"  [{p.confidence.value}]"
            )
        print("=" * 60)
        return WriteBackReport()

    from neo4j import GraphDatabase

    report = WriteBackReport()
    enriched_at = datetime.now(timezone.utc).isoformat()

    t0 = time.time()
    driver = GraphDatabase.driver(neo4j_url, auth=(neo4j_user, neo4j_pass))

    try:
        driver.verify_connectivity()
    except Exception as e:
        driver.close()
        raise ConnectionError(f"Failed to connect to Neo4j at {neo4j_url}: {e}") from e

    try:
        with driver.session() as session:
            for p in proposals:
                try:
                    query = _build_merge_query(p)
                except ValueError as e:
                    report.skipped.append(f"Invalid identifier: {e}")
                    print(f"  [SKIP] Invalid identifier: {e}")
                    continue

                src_exists = session.execute_read(
                    _check_node_exists,
                    p.source_node.label, p.source_node.key_property, p.source_node.key_value,
                )
                tgt_exists = session.execute_read(
                    _check_node_exists,
                    p.target_node.label, p.target_node.key_property, p.target_node.key_value,
                )

                if not src_exists:
                    msg = f"Source not found: {p.source_node.label}:{p.source_node.key_value}"
                    report.skipped.append(msg)
                    print(f"  [SKIP] {msg}")
                    continue
                if not tgt_exists:
                    msg = f"Target not found: {p.target_node.label}:{p.target_node.key_value}"
                    report.skipped.append(msg)
                    print(f"  [SKIP] {msg}")
                    continue

                params = {
                    "source_value": p.source_node.key_value,
                    "target_value": p.target_node.key_value,
                    "confidence": p.confidence.value,
                    "source_document": p.source_document,
                    "extracted_phrase": p.extracted_phrase,
                    "enriched_at": enriched_at,
                }

                record = session.execute_write(_execute_merge, query, params)

                if record is None:
                    msg = (f"MERGE returned no result: "
                           f"({p.source_node.label}:{p.source_node.key_value})"
                           f"-[{p.relationship_type}]->"
                           f"({p.target_node.label}:{p.target_node.key_value})")
                    report.skipped.append(msg)
                    print(f"  [SKIP] {msg}")
                    continue

                if record["was_created"]:
                    report.created += 1
                    tag = "CREATED"
                else:
                    report.updated += 1
                    tag = "UPDATED"

                print(
                    f"  [{tag}] ({p.source_node.label}:{p.source_node.key_value})"
                    f"-[{p.relationship_type}]->"
                    f"({p.target_node.label}:{p.target_node.key_value})"
                )
    finally:
        driver.close()

    elapsed = time.time() - t0
    print(f"\n  Results: {report.created} created, {report.updated} updated, "
          f"{len(report.skipped)} skipped  ({elapsed:.1f}s)")
    print("=" * 60)
    return report

In [ ]:
# ─── Validation ───────────────────────────────────────────────────────


def validate_enrichment(neo4j_url: str, neo4j_user: str, neo4j_pass: str) -> dict:
    """Query Neo4j for enriched relationships and print a summary."""
    from neo4j import GraphDatabase

    print("\n" + "=" * 60)
    print("VALIDATING ENRICHMENT")
    print("=" * 60)

    driver = GraphDatabase.driver(neo4j_url, auth=(neo4j_user, neo4j_pass))
    results = {}

    with driver.session() as session:
        # Find all enriched relationships (those with enriched_at property)
        records = session.run("""
            MATCH (src)-[r]->(tgt)
            WHERE r.enriched_at IS NOT NULL
            RETURN
                labels(src)[0] AS source_label,
                type(r) AS relationship_type,
                labels(tgt)[0] AS target_label,
                r.confidence AS confidence,
                r.source_document AS source_document,
                r.extracted_phrase AS extracted_phrase,
                r.enriched_at AS enriched_at
            ORDER BY r.enriched_at DESC
        """).data()

        results["enriched_relationships"] = records

        if records:
            print(f"\n  Found {len(records)} enriched relationship(s):\n")
            for r in records:
                print(f"    ({r['source_label']})-[{r['relationship_type']}]->({r['target_label']})")
                print(f"      confidence: {r['confidence']}")
                print(f"      source: {r['source_document']}")
                if r['extracted_phrase']:
                    phrase = r['extracted_phrase'][:80]
                    print(f"      evidence: \"{phrase}...\"")
                print()
        else:
            print("\n  No enriched relationships found.")

        # Count by relationship type
        type_counts = session.run("""
            MATCH ()-[r]->() WHERE r.enriched_at IS NOT NULL
            RETURN type(r) AS type, count(r) AS count
            ORDER BY count DESC
        """).data()

        if type_counts:
            print("  Enriched relationship counts by type:")
            for tc in type_counts:
                print(f"    {tc['type']:<25} {tc['count']:>4}")

        results["type_counts"] = type_counts

    driver.close()
    print("=" * 60)
    return results